## Why does `push` return a `Result`? (push-fails)
The `push` method from `Vec` in the standard library has no return value, but the `push` method from our `StackVec` does: it returns a `Result` indicating that it can fail. 


Why can `StackVec::push()` fail where `Vec::push()` does not?

#### Answer

In Rust, the `Vec` type dynamically resizes its allocated memory when needed, allowing it to grow as elements are added. This ensures that `Vec::push()` never fails under normal conditions. However, `StackVec` operates with a fixed-size memory buffer, meaning it cannot grow beyond its predefined capacity. When attempting to push an element into a full `StackVec`, if the memory buffer is already full, then the operation fails.

<hr>

## Why is the `'a` bound on `T` required? (lifetime)

~~~Rust
struct StackVec<'a, T> { buffer: &'a mut [T], len: usize }
~~~


Rust automatically enforces the bound `T: 'a` and will complain if type `T` lives shorter than the lifetime `'a`. For instance, if `T` is `&'b str` and `'b` is strictly shorter than `'a`, Rust won’t allow you to create the instance of `StackVec<'a, &'b str>`.

Why is the bound required? What could go wrong if the bound wasn’t enforced by Rust?

#### Answer

The `'a` bound on `T` is required because `StackVec<'a, T>` contains a reference with the lifetime `'a`, meaning any type `T` stored in the buffer must also be valid for at least `'a`. If Rust didn’t enforce this, it would be possible to create a `StackVec` holding values that might not live as long as the `StackVec` itself, leading to dangling references and undefined behavior. By enforcing `T: 'a`, Rust ensures that the type outlives the `StackVec`, preventing memory safety issues.

<hr>

## Why does `StackVec` require `T: Clone` to `pop()`? (clone-for-pop)

The `pop` method from `Vec<T>` in the standard library is implemented for all `T`, but the `pop` method from our `StackVec` is only implemented when `T` implements the `Clone` trait. 

Why might that be? What goes wrong when the bound is removed?

#### Answer

In `Vec<T>`, elements are stored on the heap, allowing them to be moved out safely when `pop()` is called. However, `StackVec` uses a fixed-size array, which resides on the stack. Rust does not allow moving elements out of an array directly because it would leave uninitialized memory behind. To work around this, `StackVec::pop()` requires `T: Clone` so that it can return a cloned value instead of attempting to move it. Without this bound, popping an element would be unsafe, as Rust prevents partial moves from arrays to maintain memory safety.

<hr>

## Which tests make use of the `Deref` implementations? (deref-in-tests)
Read through the tests we have provided in `src/tests.rs`. 

Which tests would fail to compile if the `Deref` implementation did not exist? What about the `DerefMut` implementation? Why?

#### Answer

- The tests that use indexing (`stack_vec[i]`) and iteration (`stack_vec.iter()`) rely on `Deref` and sometimes `DerefMut` to convert `StackVec` into a slice (`&[T]` or `&mut [T]`), which provides these operations. 
- Without a `Deref` implementation, indexing and iteration tests would fail because `StackVec` itself does not directly support these methods. 
- `DerefMut` enables mutable indexing (`stack_vec[i] = value`). 
- If only `Deref` were implemented, mutable indexing tests would still fail, as `DerefMut` is required for mutable access.

<hr>